# TrustVision — Notebook maître & démonstrateur (S0 → S4)

Point d'entrée unique du projet : il relie les livrables des trois membres sans réentraîner les modèles officiels.

| Onglet | Membre | Contenu | Artefacts |
| :--- | :--- | :--- | :--- |
| **1 · CLIP & Attention** | A | Zero-shot par prompts + attention CLS → patches | `results/s0_clip.json`, `src/clip_attn.py` |
| **2 · SimCLR** | C | Courbes 1/5/10/100 % + inférence live par budget | `results/s1_simclr.json`, `checkpoints/s1_*` |
| **3 · XAI & FGSM** | B | Température, FGSM, Grad-CAM, Integrated Gradients | `checkpoints/cnn_*_seed42.pt`, `src/gradio_tab3.py` |
| **4 · EWC** | C | Curseur lambda + old/new/forgetting + inférence | `results/s4_ewc.json`, `checkpoints/ewc_*` |

**Règle du projet : aucune valeur simulée.** Toutes les métriques affichées sont lues dans les JSON produits par les runs réels ; le démonstrateur ne fait que recharger des checkpoints.

**Ordre d'exécution conseillé** : sections 0 → 1 → 2 → 3 → 4, puis lancer Gradio (section 5). Les sections 6–8 sont optionnelles.


## 0. Préparation Colab

Le notebook suppose que l'archive `TrustVision_colab.zip` est présente dans `/content` ou dans `MyDrive`.  
Elle contient le code, les JSON et les checkpoints, mais **pas** le dataset (téléchargé automatiquement à l'étape 1).

In [ ]:
# 0.1 — Monter Google Drive (optionnel, utile pour sauvegarder les artefacts)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive non monté (normal hors Colab) : {exc}')


In [ ]:
# 0.2 — Extraire l'archive et se placer à la racine du dépôt
import os
import zipfile
from pathlib import Path

PROJECT = Path('/content/TrustVision')
ZIP_CANDIDATES = [
    Path('/content/drive/MyDrive/TrustVision_colab.zip'),
    Path('/content/TrustVision_colab.zip'),
]

if not (PROJECT / 'src').exists():
    for candidate in ZIP_CANDIDATES:
        if candidate.exists():
            print(f'Extraction de {candidate} ...')
            with zipfile.ZipFile(candidate) as zf:
                zf.extractall('/content')
            break
    else:
        print('Archive absente : uploade TrustVision_colab.zip dans /content ou MyDrive.')

os.chdir(PROJECT if PROJECT.exists() else Path.cwd())
print('Répertoire de travail :', Path.cwd())


In [ ]:
# 0.3 — Installer uniquement les paquets manquants (Gradio épinglé)
import importlib.util
import subprocess
import sys


def pip_install(*specs: str) -> None:
    print("Installation de", ", ".join(specs), "...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *specs])


def ensure(module_name: str, pip_spec: str | None = None) -> None:
    if importlib.util.find_spec(module_name) is None:
        pip_install(pip_spec or module_name)


ensure("transformers")
ensure("pytest")
ensure("ipywidgets")

try:
    import gradio as gr
    if not gr.__version__.startswith("4."):
        raise ImportError(gr.__version__)
except Exception:
    pip_install(
        "gradio==4.44.1",
        "gradio_client==1.3.0",
        "pydantic==2.10.6",
        "fastapi==0.115.6",
        "starlette==0.41.3",
    )
    import gradio as gr

import fastapi
import pydantic

print("gradio   ", gr.__version__)
print("pydantic ", pydantic.VERSION)
print("fastapi  ", fastapi.__version__)


In [ ]:
# 0.4 — Versions réellement utilisées et vérification des artefacts
import platform
import PIL
import matplotlib
import numpy as np
import sklearn
import torch
import torchvision

print('Python     :', platform.python_version())
print('torch      :', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('torchvision:', torchvision.__version__)
print('numpy      :', np.__version__)
print('sklearn    :', sklearn.__version__)
print('Pillow     :', PIL.__version__)
print('matplotlib :', matplotlib.__version__)

required = [
    'src/gradio_app.py',
    'results/s0_clip.json',
    'results/s1_simclr.json',
    'results/s2_calibration.json',
    'results/s3_adversarial.json',
    'results/s4_ewc.json',
    'results/cross_effects_table.json',
    'results/clip_attn_test.npy',
    'checkpoints/cnn_base_seed42.pt',
    'checkpoints/s1_simclr_probe_1pct.pt',
    'checkpoints/s1_scratch_1pct.pt',
    'checkpoints/ewc_plus_replay_seed42.pt',
]
missing = [path for path in required if not Path(path).exists()]
print('\nArtefacts manquants :', missing or 'aucun')


## 1. Contrat de données & tests

Le dataset NEU-CLS (1 800 images, 6 classes, 300/classe) est téléchargé automatiquement (KaggleHub puis miroir Figshare). Le split est verrouillé par `configs/splits.json` : 1 260 train / 180 calibration / 360 test, vérifié par SHA-256.

In [ ]:
# 1.1 — Télécharger / vérifier le dataset et le split partagé
import hashlib
from src.data import get_dataloaders

train_loader, cal_loader, test_loader, _ = get_dataloaders(
    root_dir='data/NEU-CLS', batch_size=32, seed=42, img_size=96, auto_download=True
)
print(f'Train: {len(train_loader.dataset)} | Cal: {len(cal_loader.dataset)} | Test: {len(test_loader.dataset)}')
print('SHA-256 splits.json :', hashlib.sha256(Path('configs/splits.json').read_bytes()).hexdigest())


In [ ]:
# 1.2 — Tests Membre C (22 attendus) puis suite complète
!pytest tests/test_exp_c.py -q
print('\n--- Suite complète ---')
!pytest tests/ -q


## 2. Matrice d'effets croisés S0 → S4

Vue consolidée des cinq étapes : exactitude, calibration, robustesse FGSM, similarité XAI et oubli catastrophique.

In [ ]:
# 2.1 — Tableau comparatif
import json
import pandas as pd

table = json.loads(Path('results/cross_effects_table.json').read_text(encoding='utf-8'))
df = pd.DataFrame(table['rows'])
columns = ['stage', 'acc', 'macro_f1', 'ece', 'nll', 'brier', 'rob_acc_003', 'xai_sim', 'intra_xai_stab', 'forgetting']
display(df[[c for c in columns if c in df.columns]].rename(columns={
    'stage': 'Étape', 'acc': 'Acc', 'macro_f1': 'Macro-F1', 'ece': 'ECE',
    'nll': 'NLL', 'brier': 'Brier', 'rob_acc_003': 'Rob. FGSM',
    'xai_sim': 'XAI Sim', 'intra_xai_stab': 'Stab. intra-XAI', 'forgetting': 'Forgetting',
}))


## 3. S1 — Efficacité des étiquettes (SimCLR)

Les sondes SimCLR sont des régressions logistiques standardisées sur les features gelées, calibrées sur le split de calibration partagé. La baseline « scratch » est un CNN entraîné sur les mêmes sous-ensembles d'étiquettes.

In [ ]:
# 3.1 — Courbes réelles et tableau 1/5/10/100 %
import matplotlib.pyplot as plt

s1 = json.loads(Path('results/s1_simclr.json').read_text(encoding='utf-8'))
keys = ['1pct', '5pct', '10pct', '100pct']
xs = [int(k.replace('pct', '')) for k in keys]
sim = [s1['simclr']['label_fractions'][k]['acc'] * 100 for k in keys]
scratch = [s1['supervised_from_scratch']['label_fractions'][k]['acc'] * 100 for k in keys]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=130)
axes[0].plot(xs, sim, marker='o', linewidth=2, label='SimCLR (sonde gelée)')
axes[0].plot(xs, scratch, marker='s', linewidth=2, label='CNN from scratch')
axes[0].set_xscale('log'); axes[0].set_xticks(xs); axes[0].set_xticklabels([f'{x}%' for x in xs])
axes[0].set_xlabel("Budget d'étiquettes"); axes[0].set_ylabel('Accuracy test (%)')
axes[0].set_title('S1 — Efficacité des étiquettes', fontweight='bold'); axes[0].grid(alpha=0.3); axes[0].legend()

sim_ece = [(s1['simclr']['label_fractions'][k].get('calibrated') or {}).get('ece', s1['simclr']['label_fractions'][k]['ece']) for k in keys]
scratch_ece = [(s1['supervised_from_scratch']['label_fractions'][k].get('calibrated') or {}).get('ece', s1['supervised_from_scratch']['label_fractions'][k]['ece']) for k in keys]
axes[1].plot(xs, sim_ece, marker='o', linewidth=2, label='SimCLR (calibré)')
axes[1].plot(xs, scratch_ece, marker='s', linewidth=2, label='Scratch (calibré)')
axes[1].set_xscale('log'); axes[1].set_xticks(xs); axes[1].set_xticklabels([f'{x}%' for x in xs])
axes[1].set_xlabel("Budget d'étiquettes"); axes[1].set_ylabel('ECE (bas = mieux)')
axes[1].set_title('S1 — Calibration', fontweight='bold'); axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.show()

print('Modèle principal S1 :', s1['main_model']['source'], f"({s1['main_model']['acc'] * 100:.2f} %)")
print('XAI Sim S1 :', s1.get('xai_sim'))


In [ ]:
# 3.2 — Intervalles de confiance multi-seeds (si le run a été exécuté)
multiseed_path = Path('results/s1_multiseed_ci.json')
if multiseed_path.exists():
    multi = json.loads(multiseed_path.read_text(encoding='utf-8'))
    rows = []
    for key in multi['fractions']:
        for method, label in (('simclr', 'SimCLR'), ('scratch', 'Scratch')):
            stats = multi['aggregates'][key][method]['acc']
            rows.append({
                'Budget': key, 'Méthode': label,
                'Moyenne %': round(stats['mean'] * 100, 2),
                'Écart-type %': round(stats['std'] * 100, 2),
                'IC95 bas %': round(stats['ci95_low'] * 100, 2),
                'IC95 haut %': round(stats['ci95_high'] * 100, 2),
            })
    display(pd.DataFrame(rows))
    print(multi['ci_method'])
else:
    print('Non exécuté. Commande : python -m scripts.run_s1_multiseed')


## 4. S4 — Apprentissage continu (EWC)

Scénario **single-head 3 → 6 classes sans identifiant de tâche**. L'encodeur SimCLR a été pré-entraîné uniquement sur les 3 classes initiales. Le tableau compare le fine-tuning naïf, EWC pur (balayage lambda), replay et EWC + replay.

In [ ]:
# 4.1 — Tableau des stratégies et sweep lambda
s4 = json.loads(Path('results/s4_ewc.json').read_text(encoding='utf-8'))
rows = []
for name, strategy in s4['strategies'].items():
    metrics = strategy['metrics']
    rows.append({
        'Stratégie': name, 'Lambda': strategy.get('ewc_lambda', 0),
        'Acc %': round(metrics['acc'] * 100, 2),
        'Old %': round(metrics['old_acc'] * 100, 2),
        'New %': round(metrics['new_acc'] * 100, 2),
        'Forgetting %': round(metrics['forgetting'] * 100, 2),
        'ECE': round(metrics['ece'], 4),
    })
display(pd.DataFrame(rows))

ewc_keys = sorted([k for k in s4['strategies'] if k.startswith('ewc_lambda_')], key=lambda k: s4['strategies'][k]['ewc_lambda'])
lambdas = [s4['strategies'][k]['ewc_lambda'] for k in ewc_keys]
fig, ax = plt.subplots(figsize=(8.4, 4), dpi=130)
for metric, label, marker in (('old_acc', 'Anciennes classes', 'o'), ('new_acc', 'Nouvelles classes', 's'), ('acc', 'Global', '^')):
    ax.plot(lambdas, [s4['strategies'][k]['metrics'][metric] * 100 for k in ewc_keys], marker=marker, linewidth=2, label=label)
ax.axhline(s4['strategies']['replay']['metrics']['acc'] * 100, linestyle='--', color='gray', label='Replay seul (global)')
ax.set_xscale('log'); ax.set_xlabel('Lambda EWC'); ax.set_ylabel('Accuracy (%)')
ax.set_title('S4 — Balayage lambda (run 3)', fontweight='bold'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print('Meilleure variante EWC :', s4['best_ewc']['strategy'], '| lambda =', s4['best_ewc']['ewc_lambda'])
print('Replay seul :', s4['strategies']['replay']['metrics']['acc'] * 100, '% (meilleur modèle S4 toutes familles)')
print('XAI Sim S4 :', s4.get('xai_sim'))


## 5. Démonstrateur Gradio — 4 onglets

Le module `src/gradio_app.py` assemble :

1. **CLIP & Attention (A)** — le premier clic télécharge CLIP (~600 Mo).
2. **SimCLR (C)** — courbes + inférence avec les checkpoints 1/5/10/100 %.
3. **XAI & FGSM (B)** — température, epsilon, Grad-CAM, Integrated Gradients.
4. **EWC (C)** — curseur lambda qui charge `ewc_lambda_*_seed42.pt` et affiche les métriques réelles.

Sur Colab, `share=True` produit un lien public valable 72 h.

In [ ]:
# 5.1 — Lancer le démonstrateur
from src.gradio_app import build_demo

demo = build_demo()
demo.launch(share=True, debug=False)


## 6. Mini-TP — lire une stratégie S4

Manipule le menu déroulant pour comparer les stratégies sans réentraînement : chaque entrée pointe vers le checkpoint et les métriques du run 3.

In [ ]:
# 6.1 — Widget de lecture des stratégies S4
import ipywidgets as widgets
from IPython.display import Markdown, display

options = list(s4['strategies'].keys())
selector = widgets.Dropdown(options=options, value='ewc_replay', description='Stratégie :')
output = widgets.Output()


def show_strategy(name):
    strategy = s4['strategies'][name]
    metrics = strategy['metrics']
    calibrated = metrics.get('calibrated') or {}
    accepted = calibrated.get('calibration_accepted', True)
    text = (
        f"### {name} (lambda = {strategy.get('ewc_lambda', 0):g})\n"
        f"- Accuracy : **{metrics['acc'] * 100:.2f} %**\n"
        f"- Old (1-3) : {metrics['old_acc'] * 100:.2f} % | New (4-6) : {metrics['new_acc'] * 100:.2f} %\n"
        f"- Forgetting : {metrics['forgetting'] * 100:.2f} %\n"
        f"- ECE : {metrics['ece']:.4f} ({'calibrée' if accepted else 'brute, fit rejeté'})\n"
        f"- Robustesse FGSM eps=0.03 : {metrics['rob_acc_003'] * 100:.2f} %\n"
        f"- Checkpoint : `{strategy.get('checkpoint')}`"
    )
    with output:
        output.clear_output()
        display(Markdown(text))


def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        show_strategy(change['new'])


selector.observe(on_change)
display(selector, output)
show_strategy('ewc_replay')


## 7. Reproductibilité complémentaire

Ces commandes sont déjà exécutées dans le dépôt ; elles sont rappelées pour un audit indépendant.

```bash
# Export des checkpoints 1/5/10 % (sans toucher aux 100 % officiels)
python -m scripts.export_s1_fraction_checkpoints

# IC 95 % multi-seeds (42, 123, 999) à 1 % et 5 %
python -m scripts.run_s1_multiseed

# Recalcul des XAI Sim S1/S4 avec le protocole S2
python -m scripts.compute_xai_sim
```


In [ ]:
# 7.1 — Inventaire des checkpoints et vérification rapide
import torch

inventory = []
for path in sorted(Path('checkpoints').glob('*.pt')):
    state = torch.load(path, map_location='cpu', weights_only=False)
    classes = state['fc.weight'].shape[0] if isinstance(state, dict) and 'fc.weight' in state else 'n/a'
    inventory.append({'Checkpoint': path.name, 'Classes': classes, 'Taille (Mo)': round(path.stat().st_size / 1e6, 2)})
display(pd.DataFrame(inventory))


## 8. Sauvegarde des artefacts

Copie les résultats et figures dans Drive pour archivage. Les checkpoints restent dans l'archive de départ ; seuls les JSON et figures sont recopiés.

In [ ]:
# 8.1 — Archive de session (si Drive est monté)
import shutil
from datetime import datetime

if Path('/content/drive/MyDrive').exists():
    stamp = datetime.now().strftime('%Y%m%d_%H%M')
    archive = Path(f'/content/drive/MyDrive/trustvision_session_{stamp}')
    archive.mkdir(parents=True, exist_ok=True)
    shutil.copytree('results', archive / 'results', dirs_exist_ok=True)
    print('Artefacts copiés vers', archive)
else:
    print('Drive non monté : rien à sauvegarder.')


## Checklist de validation

- [ ] 22/22 tests Membre C passent.
- [ ] `results/s1_multiseed_ci.json` contient 3 seeds et l'IC 95 % à 1 % et 5 % (t = 4,303, n = 3).
- [ ] Le split SHA-256 correspond à `s0_clip.json` (`2848c9ae…`).
- [ ] La matrice S0–S4 contient les valeurs réelles, dont `XAI Sim` S1 = 0.7118 et S4 = 0.7961.
- [ ] L'onglet 1 affiche l'attention CLIP et rappelle le score zero-shot (25,56 %) vs la sonde calibrée (99,44 %).
- [ ] L'onglet 2 charge les checkpoints 1/5/10/100 % et affiche les courbes réelles.
- [ ] L'onglet 3 respecte le contrat de latence < 500 ms.
- [ ] L'onglet 4 charge chaque `ewc_lambda_*` et affiche old/new/forgetting.
- [ ] Le **replay seul** (93,06 %) est présenté comme le meilleur modèle S4, pas l'hybride EWC.
